# 01b - Extended Data Validation
**Owner:** Member 2 (Dhananjay)

Builds on `01_data_validation.ipynb` (missing values, duplicate rows, data types). This notebook adds the checks the causal models depend on: one row per region per day, revenue consistency, outliers, and a clear PASS/FAIL record.

In [1]:
import pandas as pd

df = pd.read_csv("project1_algorithmic_pricing_synthetic_dataset.csv", parse_dates=["date"])
numeric_cols = ["our_price", "competitor_price", "units_sold", "revenue", "marketing_spend"]
print("Rows, columns:", df.shape)

Rows, columns: (540, 11)


## 1. One row per region per day
Synthetic Control compares regions day by day, so every region needs every day exactly once.

In [2]:
key_dupes = int(df.duplicated(subset=["date", "region", "product_category"]).sum())
print("Duplicate date-region rows:", key_dupes)

days = df.groupby("region")["date"].agg(["min", "max", "nunique"])
days["expected_days"] = (days["max"] - days["min"]).dt.days + 1
days["missing_days"] = days["expected_days"] - days["nunique"]
days

Duplicate date-region rows: 0


,min,max,nunique,expected_days,missing_days
region,,,,,
North,2025-01-01,2025-06-29,180,180,0
South,2025-01-01,2025-06-29,180,180,0
West,2025-01-01,2025-06-29,180,180,0


## 2. Revenue consistency
Revenue should equal our price x units sold. A mismatch would mean the revenue column cannot be trusted as the outcome.

In [3]:
diff = (df["our_price"] * df["units_sold"] - df["revenue"]).abs()
revenue_mismatch = int((diff > 1).sum())
print("Rows where revenue != price x units (tolerance Rs 1):", revenue_mismatch)
print("Largest difference:", round(diff.max(), 2))

Rows where revenue != price x units (tolerance Rs 1): 0
Largest difference: 0.79


## 3. Outliers (IQR rule)
Values far outside the normal range can distort treatment effect estimates. Flagged values are reviewed, not deleted.

In [4]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()

outliers = df.groupby("region")[numeric_cols].agg(iqr_outliers)
outliers

,our_price,competitor_price,units_sold,revenue,marketing_spend
region,,,,,
North,0,0,1,2,2
South,3,0,5,5,5
West,4,0,3,3,2


## 4. Treatment timing and price change

In [5]:
print(pd.crosstab(df["region"], df["treatment"]))
print("Treatment starts:", df.loc[df["treatment"] == 1, "date"].min().date())
period_mismatch = int(((df["treatment"] == 1) & (df["period"] != "post_treatment")).sum())
print("Treated rows outside post_treatment:", period_mismatch)

df.groupby(["region", "period"])[["our_price", "units_sold", "revenue"]].mean().round(1)

treatment    0   1
region            
North      120  60
South      180   0
West       180   0
Treatment starts: 2025-05-01
Treated rows outside post_treatment: 0


our_price  units_sold  revenue
region period                                        
North  post_treatment      534.6       124.2  66366.4
       pre_treatment       500.0       123.9  61944.4
South  post_treatment      489.6       118.3  57908.5
       pre_treatment       490.0       114.7  56217.2
West   post_treatment      510.2       136.7  69749.2
       pre_treatment       509.9       130.4  66477.0

## 5. Validation summary

In [6]:
summary = pd.DataFrame({
    "check": ["Missing values", "Exact duplicate rows", "Duplicate date-region rows",
              "No missing days", "No zero/negative values", "Revenue = price x units",
              "Treatment only in post_treatment"],
    "result": [df.isna().sum().sum() == 0, df.duplicated().sum() == 0, key_dupes == 0,
               (days["missing_days"] == 0).all(), bool(((df[numeric_cols] <= 0).sum() == 0).all()),
               revenue_mismatch == 0, period_mismatch == 0],
})
summary["result"] = summary["result"].map({True: "PASS", False: "FAIL"})
summary

,check,result
0,Missing values,PASS
1,Exact duplicate rows,PASS
2,Duplicate date-region rows,PASS
3,No missing days,PASS
4,No zero/negative values,PASS
5,Revenue = price x units,PASS
6,Treatment only in post_treatment,PASS


## Findings
- All 7 checks pass: 540 rows, 3 regions x 180 days (1 Jan to 29 Jun 2025), no gaps or duplicates.
- Revenue equals price x units on every row, so it is safe to use as the outcome variable.
- Only **North** is treated: its average price rises from about 500 to about 535 from 1 May 2025. South and West stay flat.
- South and West form the donor pool for Synthetic Control. Two donors is very few, which limits how well a synthetic North can be built; flagged for the mentor review.
- See the outlier table above; flagged values are kept, since a promotion day can legitimately push sales up.